## Integrating External MCP Servers

# Lesson 6: Protocol Extensions and Modular Tools via Model Context Protocol (MCP)

In the previous lesson, you mastered maintaining agent state using `ClaudeSDKClient`, allowing you to build persistent, multi-turn conversational systems. Throughout those modules, you leveraged built-in tools like `WebSearch`, `Bash`, `Read`, and `Write`.

While these built-in utilities handle common operational tasks, real-world engineering environments often require specialized capabilities—such as querying production databases, inspecting local Git abstractions, or running semantic code indexes.

This is where **MCP (Model Context Protocol) servers** come in. MCP is an open-standard architectural layer that lets you connect external plug-and-play tool packages to your agent with minimal code setup.

---

## What Are MCP Servers?

An MCP server is an out-of-process package that exposes specialized tools to an AI agent. Think of them as plugins or extensions. For example, the `Context7` MCP server grants your agent instant semantic documentation search access across thousands of open-source libraries, while other community servers provide direct connections to PostgreSQL databases, GitHub enterprise APIs, or local file parsers.

### Architectural Separations:

* **Process Isolation:** Every MCP server runs as a separate, isolated process on your machine or a remote host. It communicates with the Claude Agent SDK via standard, unified JSON-RPC structures.
* **Extensibility:** Because the protocol decouples tool definitions from the core model framework, you can inject entirely new capabilities into your agent without modifying the SDK codebase.

---

## Declaring an MCP Server Configuration

To bind an external MCP server to your execution loop, define a configuration dictionary specifying the communication protocol, the execution engine command, and the arguments required to initialize the server process:

```python
# Define external MCP server configuration
context7_config = {
    "type": "stdio",  # Communicates via standard input/output streams
    "command": "npx", # Node.js package execution binary wrapper
    "args": ["-y", "@upstash/context7-mcp"] # Target package arguments
}

```

### Configuration Fields Decoded:

* **`"type"`:** Defines the transport layer protocol. The value `"stdio"` tells the SDK to pipe data directly through the standard input and output streams (`stdin`/`stdout`) of the spawned process, which is the most common approach for local tooling.
* **`"command"`:** The executable binary invoked by the host system. Using `"npx"` allows Node.js package environments to download and initialize tools on the fly without global installations.
* **`"args"`:** A list of string arguments passed directly to the execution command. The `"-y"` flag auto-approves installation prompts, keeping process spin-up headless and non-blocking.

---

## Connecting External Servers via `mcp_servers`

Once you declare your server configuration dictionary, map it into your active agent workspace by registering it within the `mcp_servers` parameter inside `ClaudeAgentOptions`:

```python
from claude_agent_sdk import ClaudeAgentOptions

options = ClaudeAgentOptions(
    model="haiku",
    max_turns=10,
    # Connect the external server via a unique key identifier
    mcp_servers={
        "context7": context7_config
    }
)

```

The `mcp_servers` dictionary maps developer-defined string keys (like `"context7"`) directly to their respective process configuration objects. You can connect multiple servers simultaneously by expanding this dictionary dictionary map.

---

## Whitelisting MCP Tools

Connecting an MCP server makes its tool ecosystem visible to the runtime, but for security and cost containment, **the agent cannot execute them until you explicitly declare them inside your `allowed_tools` register.**

MCP tools use a namespaced string syntax to prevent collisions with built-in tools:

$$\text{mcp\_\_}\langle\text{server\_key}\rangle\text{\_\_}\langle\text{tool\_name}\rangle$$

```python
options = ClaudeAgentOptions(
    model="haiku",
    max_turns=10,
    mcp_servers={
        "context7": context7_config
    },
    # Whitelist the tools provided by this server
    allowed_tools=[
        "mcp__context7__resolve-library-id", # Tool 1: Searches library hashes
        "mcp__context7__get-library-docs"     # Tool 2: Extracts full doc strings
    ]
)

```

### Breaking Down the Namespace Elements:

* **`mcp__`:** The structural prefix identifying the utility as an external protocol tool rather than a built-in function.
* **`context7`:** Matches the unique dictionary key assigned inside the `mcp_servers` block.
* **`resolve-library-id`:** The precise tool name exposed by the target MCP server application.

---

## Guiding Claude to Use MCP Tools

While Claude is capable of choosing its own tools based on schema definitions, it won't automatically know when to prioritize an external protocol server over a standard internet search. Use clear, explicit framing in your prompt string to direct the agent toward your new tools:

```python
async with ClaudeSDKClient(options=options) as client:
    # Guide the agent explicitly to use the whitelisted MCP namespace
    await client.query(
        "Using the context7 tools, search for documentation on how the "
        "Claude Agent SDK (Python) uses MCP servers. Summarize what you find."
    )

```

---

## Complete Blueprint: Integration Script

Combining the configuration, registries, and stateful sessions yields the following unified automation blueprint:

```python
import anyio
from claude_agent_sdk import ClaudeSDKClient, ClaudeAgentOptions
from utils import display_response # Reusable streaming utility

async def main():
    # 1. Define external MCP server configuration
    context7_config = {
        "type": "stdio",
        "command": "npx",
        "args": ["-y", "@upstash/context7-mcp"]
    }
    
    # 2. Package options and map namespaced tools
    options = ClaudeAgentOptions(
        model="haiku",
        max_turns=10,
        mcp_servers={
            "context7": context7_config
        },
        allowed_tools=[
            "mcp__context7__resolve-library-id",
            "mcp__context7__get-library-docs"
        ]
    )
    
    # 3. Instantiate stateful runtime container context
    async with ClaudeSDKClient(options=options) as client:
        await client.query(
            "Using the context7 tools, search for documentation on how the "
            "Claude Agent SDK uses MCP servers. Summarize what you find."
        )
        await display_response(client)

if __name__ == "__main__":
    anyio.run(main)

```

---

## Diagnostic Multi-Turn Execution Trace

When running this code, watch your logs to trace how the agent chains your custom protocol tools together to extract information:

### Step 1: Library Resolution (`resolve-library-id`)

The agent reads your prompt, bypasses standard internet search, and routes the query directly to the namespaced MCP tool to pinpoint the exact library target:

```text
💬 Claude Response:
I will search for official documentation on how the Claude Agent SDK integrates MCP servers using the connected Context7 tools.

🔧 [Tool: mcp__context7__resolve-library-id]

```

### Step 2: Documentation Retrieval (`get-library-docs`)

The server process returns structural identifiers matching the query. The agent parses this payload and automatically requests the underlying documentation layout using the second whitelisted tool:

```text
💬 Claude Response:
I found several records. The most comprehensive resource is `/websites/platform_claude_en_agent-sdk` which outlines deep MCP server configurations. Let me pull the raw content layouts:

🔧 [Tool: mcp__context7__get-library-docs]

```

### Step 3: Synthesis and Structured Output Generation

With the external documentation ingested into its conversational memory state, Claude builds a comprehensive technical summary:

```text
💬 Claude Response:
## Summary: Claude Agent SDK and MCP Servers

Based on the documentation I pulled, here is the structural breakdown of how the Claude Agent SDK manages MCP integrations:

### Core Integration Methods
1. **In-Process SDK MCP Servers:** Built inside the same host process via the `@tool` decorator and `create_sdk_mcp_server()`.
2. **External MCP Servers:** Processes configured via dictionaries passed to `ClaudeAgentOptions` that handle JSON-RPC messaging.

### Transport Protocols Supported
* **Stdio:** Standard input/output communication channels (`type: "stdio"`).
* **HTTP & SSE:** Remote Server-Sent Events with customizable authorization headers.

```

---

## Summary

You have unlocked modular agent architecture capabilities using the Model Context Protocol:

* The **`mcp_servers`** property isolates external tool logic from your primary core script loop.
* The namespaced **`allowed_tools`** string schema ($mcp\_\_\langle\text{server}\rangle\_\_\langle\text{tool}\rangle$) maintains strict operational sandboxing.
* Framing prompts with **explicit tool directives** ensures the agent efficiently routes calls to specialized extensions instead of generic fallback mechanisms.

Let's head into the practice labs to boot up your first external MCP server process! 🚀

## Guiding Your Agent Towards MCP Tools

Now that you've seen how MCP servers extend your agent's capabilities, it's time to practice an important skill: writing prompts that actually guide your agent to use those external tools effectively.

In this exercise, you have a fully configured agent with the Context7 MCP server connected and its tools whitelisted. However, the prompt is currently empty, so the agent won't know what to do or which tools to use.

Your job is to write a prompt that:

    Explicitly mentions context7 tools to guide the agent toward using the MCP tools
    Asks the agent to search for documentation
    Specifies a topic to search for (such as how the Claude Agent SDK uses MCP servers or any other technical topic you're curious about)

Remember, the agent has access to both built-in tools and MCP tools, so being explicit in your prompt helps it choose the right tools for the task. Write a clear, directive prompt and watch your agent leverage external documentation search capabilities!

```
import anyio
from claude_agent_sdk import ClaudeSDKClient, ClaudeAgentOptions
from utils import display_response


async def main():
    # 1. Define external MCP server configuration
    # Uses 'npx' to run the Context7 server directly
    context7_config = {
        "type": "stdio",  # Communicates via stdin/stdout
        "command": "npx",
        "args": ["-y", "@upstash/context7-mcp"]
    }

    # 2. Configure the agent
    options = ClaudeAgentOptions(
        model="haiku",
        max_turns=10,
        # Connect the external server
        mcp_servers={
            "context7": context7_config
        },
        # Whitelist the tools provided by this server
        # Format: mcp__<server_key>__<tool_name>
        allowed_tools=[
            "mcp__context7__resolve-library-id",
            "mcp__context7__get-library-docs"
        ]
    )
    
    # 3. Run the agent
    async with ClaudeSDKClient(options=options) as client:
        # TODO: Write a prompt that guides the agent to use the context7 MCP tools
        # Your prompt should:
        # - Explicitly mention "context7 tools" to guide the agent
        # - Ask to search for documentation
        # - Specify what topic to search for (e.g., how Claude Agent SDK uses MCP servers)
        await client.query(
            ""
        )
        
        await display_response(client)


if __name__ == "__main__":
    anyio.run(main)
```

Here is the completed code with an explicit, directive prompt added to the `client.query()` block.

By explicitly naming the `"context7 tools"`, you give the model clear instructions to route its reasoning loop through the external protocol server instead of using default internet fallbacks:

```python
import anyio
from claude_agent_sdk import ClaudeSDKClient, ClaudeAgentOptions
from utils import display_response


async def main():
    # 1. Define external MCP server configuration
    # Uses 'npx' to run the Context7 server directly
    context7_config = {
        "type": "stdio",  # Communicates via stdin/stdout
        "command": "npx",
        "args": ["-y", "@upstash/context7-mcp"]
    }

    # 2. Configure the agent
    options = ClaudeAgentOptions(
        model="haiku",
        max_turns=10,
        # Connect the external server
        mcp_servers={
            "context7": context7_config
        },
        # Whitelist the tools provided by this server
        # Format: mcp__<server_key>__<tool_name>
        allowed_tools=[
            "mcp__context7__resolve-library-id",
            "mcp__context7__get-library-docs"
        ]
    )
    
    # 3. Run the agent
    async with ClaudeSDKClient(options=options) as client:
        # Write a prompt that guides the agent to use the context7 MCP tools
        await client.query(
            "Using the context7 tools, search for documentation on how the "
            "Claude Agent SDK uses MCP servers. Summarize what you find."
        )
        
        await display_response(client)


if __name__ == "__main__":
    anyio.run(main)

```

### Prompt Engineering Breakdown for MCP:

* **The Namespace Hook:** Including the text *"Using the context7 tools"* helps the model link your request to the `mcp__context7__...` namespace tools declared inside the allowlist registry.
* **Objective-Driven Scoping:** By clarifying that it must *search for documentation* and *summarize what it finds*, the agent understands it should first execute the lookup tool (`resolve-library-id`), pull the payload text blocks (`get-library-docs`), and then finish by parsing the results into a human-readable text block response.

## Troubleshooting MCP Tool Access

In this exercise, you have a fully configured agent with the Context7 MCP server connected and a working prompt ready to go. However, when you run the code, the agent is unable to use the MCP tools provided by the server.

Your task is to investigate the configuration and make any necessary corrections so that the agent can successfully access and use the external Context7 tools. Once you’ve fixed the issue, run the code and confirm that your agent is able to search through documentation using the external server.

```
import anyio
from claude_agent_sdk import ClaudeSDKClient, ClaudeAgentOptions
from utils import display_response


async def main():
    # 1. Define external MCP server configuration
    # Uses 'npx' to run the Context7 server directly
    context7_config = {
        "type": "stdio",  # Communicates via stdin/stdout
        "command": "npx",
        "args": ["-y", "@upstash/context7-mcp"]
    }

    # 2. Configure the agent
    options = ClaudeAgentOptions(
        model="haiku",
        max_turns=10,
        # Connect the external server
        mcp_servers={
            "context7": context7_config
        },
        # Whitelist the tools provided by this server
        allowed_tools=[
            "context7__resolve-library-id",
            "context7__get-library-docs"
        ]
    )
    
    # 3. Run the agent
    async with ClaudeSDKClient(options=options) as client:
        # The prompt guides the agent to use the MCP tools
        await client.query(
            "Using the context7 tools, search for documentation on how the "
            "Claude Agent SDK (Python) uses MCP servers. Summarize what you find."
        )
        
        await display_response(client)


if __name__ == "__main__":
    anyio.run(main)
```

The issue in the starter code lies within the `allowed_tools` allowlist strings.

For the Claude Agent SDK to map external tools correctly, they must follow a strict structural namespace format: `mcp__<server_key>__<tool_name>`. The starter code is missing the critical structural `mcp__` prefix, causing the system to reject the tool calls as unauthorized.

Here is the corrected code with the proper namespacing applied to the whitelisted tools:

```python
import anyio
from claude_agent_sdk import ClaudeSDKClient, ClaudeAgentOptions
from utils import display_response


async def main():
    # 1. Define external MCP server configuration
    # Uses 'npx' to run the Context7 server directly
    context7_config = {
        "type": "stdio",  # Communicates via stdin/stdout
        "command": "npx",
        "args": ["-y", "@upstash/context7-mcp"]
    }

    # 2. Configure the agent
    options = ClaudeAgentOptions(
        model="haiku",
        max_turns=10,
        # Connect the external server
        mcp_servers={
            "context7": context7_config
        },
        # Whitelist the tools provided by this server with the correct 'mcp__' prefix
        allowed_tools=[
            "mcp__context7__resolve-library-id",
            "mcp__context7__get-library-docs"
        ]
    )
    
    # 3. Run the agent
    async with ClaudeSDKClient(options=options) as client:
        # The prompt guides the agent to use the MCP tools
        await client.query(
            "Using the context7 tools, search for documentation on how the "
            "Claude Agent SDK (Python) uses MCP servers. Summarize what you find."
        )
        
        await display_response(client)


if __name__ == "__main__":
    anyio.run(main)

```

### Troubleshooting Breakdown:

* **The Root Cause:** Without the `mcp__` prefix, the SDK searches its internal *built-in* tools registry for a tool named `context7__resolve-library-id`. Failing to find it there, the agent throws an execution permission error when trying to run it.
* **The Fix:** Changing the entries to `"mcp__context7__resolve-library-id"` and `"mcp__context7__get-library-docs"` tells the runtime exactly how to parse the request: look into the **MCP protocol layer** (`mcp__`), target the specific **external process key** (`context7`), and invoke the matching **server-side tool function**.

## Renaming Servers for Better Clarity

Now, let's tackle a common real-world scenario: making your MCP server configurations more descriptive by using user-friendly names.

Right now, your code uses the original server name ("context7") in the mcp_servers dictionary. While this matches the package name, it doesn't clearly communicate what the server does. Someone reading your code would need to look up what "context7" means, but a name like "docs_search" immediately tells them that this server provides documentation search capabilities.

Your task is to rename the server to something more descriptive and update all the places where this name is used:

    Change the server key in the mcp_servers dictionary from "context7" to a user-friendly name like "docs_search".
    Update the allowed_tools entries to use this new name in the format mcp__<new_name>__<tool_name>.
    Adjust the prompt to reference the new server name.

Remember, the server name you choose must be used consistently in both mcp_servers and allowed_tools for the agent to access the tools correctly. Once you've made these changes, run your code and watch your agent use the documentation search tools with a much clearer configuration!

```
import anyio
from claude_agent_sdk import ClaudeSDKClient, ClaudeAgentOptions
from utils import display_response


async def main():
    # 1. Define external MCP server configuration
    # Uses 'npx' to run the Context7 server directly
    context7_config = {
        "type": "stdio",  # Communicates via stdin/stdout
        "command": "npx",
        "args": ["-y", "@upstash/context7-mcp"]
    }

    # 2. Configure the agent
    options = ClaudeAgentOptions(
        model="haiku",
        max_turns=10,
        # TODO: Update the server name to be more descriptive
        mcp_servers={
            "context7": context7_config
        },
        # TODO: Update the allowed_tools to use the correct server name
        allowed_tools=[
            "mcp__context7__resolve-library-id",
            "mcp__context7__get-library-docs"
        ]
    )
    
    # 3. Run the agent
    async with ClaudeSDKClient(options=options) as client:
        # TODO: Update the prompt to reference the new server name
        await client.query(
            "Using the context7 tools, search for documentation on how the "
            "Claude Agent SDK (Python) uses MCP servers. Summarize what you find."
        )
        
        await display_response(client)


if __name__ == "__main__":
    anyio.run(main)
```

Here is the updated code with the MCP server key refactored to a more descriptive identifier (`"docs_search"`).

This demonstrates how the custom dictionary key you choose cascades down directly into both your namespaced allowlist registry and your conversational guidance prompt:

```python
import anyio
from claude_agent_sdk import ClaudeSDKClient, ClaudeAgentOptions
from utils import display_response


async def main():
    # 1. Define external MCP server configuration
    # Uses 'npx' to run the Context7 server directly
    context7_config = {
        "type": "stdio",  # Communicates via stdin/stdout
        "command": "npx",
        "args": ["-y", "@upstash/context7-mcp"]
    }

    # 2. Configure the agent
    options = ClaudeAgentOptions(
        model="haiku",
        max_turns=10,
        # Update the server name to be more descriptive (docs_search)
        mcp_servers={
            "docs_search": context7_config
        },
        # Update the allowed_tools to use the correct server name (mcp__docs_search__...)
        allowed_tools=[
            "mcp__docs_search__resolve-library-id",
            "mcp__docs_search__get-library-docs"
        ]
    )
    
    # 3. Run the agent
    async with ClaudeSDKClient(options=options) as client:
        # Update the prompt to reference the new server name
        await client.query(
            "Using the docs_search tools, search for documentation on how the "
            "Claude Agent SDK (Python) uses MCP servers. Summarize what you find."
        )
        
        await display_response(client)


if __name__ == "__main__":
    anyio.run(main)

```

### Architectural Consistency Rules:

* **The Namespace Cascade:** When you redefine the registry mapping key to `"docs_search"`, the SDK dynamically creates the new internal namespace boundaries. If you forget to modify the `allowed_tools` entries to match, the system will look for the old `context7` process bridge and fail immediately.
* **Readable Prompt Engineering:** Updating the prompt to say *"Using the docs_search tools"* ensures the model reads its current `allowed_tools` schema, links the human request directly to the matching `"mcp__docs_search__..."` keys, and correctly routes the out-of-process tool calls.

## Connecting External Documentation Search Tools